In [1]:
# Just for RCF only 

from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.cellulosic_tea import create_cellulosic_ethanol_tea

from biosteam import main_flowsheet as F
import biosteam as bst

chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7

chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])


rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

WWT = bst.create_conventional_wastewater_treatment_system('WWT', ins=F.RCF_WW_OUTS)


for unit in WWT.units:
    if hasattr(unit, 'strict_moisture_content'):
        unit.strict_moisture_content = False   # ← toggle here
    # To adjust the target moisture fraction (default 0.79 from Humbird):
    # if hasattr(unit, 'moisture_content'):
    #     unit.moisture_content = 0.6

BT = bst.facilities.BoilerTurbogenerator('BT', fuel_price=prices['CH4'])

gas_mixer= bst.Mixer('MIX_BT_gas',    ins=(WWT.outs[0], F.RCF_PSAWASTE_OUTS))

BT.ins[0] = WWT.outs[1]   # Connecting sludge to BT solids feed
BT.ins[1] = gas_mixer.outs[0]   # Connecting biogas from WW treatment and PSA waste gases from RCF




rcf_solo_system = bst.System(
    'Solo RCF system',
    path=(rcf_system, WWT),
    facilities=[gas_mixer, BT],
)
rcf_solo_system.simulate()

integrated_tea = create_cellulosic_ethanol_tea(rcf_solo_system)

print(f'The MSP for RCF crude oil is  {round(integrated_tea.solve_price(F.RCF_CRUDE_OUT), 3)} USD/kg')




c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: RCF_PUMP1> no pump type available at current power (2.45e+03 hp), head (3.36e+03 ft), kinematic viscosity (5.99e-07 m2/s), and NPSH (4.21 ft); assuming centrigugal pump
  warn(f'{repr(

The MSP for RCF crude oil is  2.064 USD/kg


c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWarning: <IsentropicCompressor: RCF_COMP1> power (1.487e-11 hp) is out of bounds (10 to 750 hp) for cost correlation
  self._cost(**cost_kwargs) if cost_kwargs else self._cost()


In [2]:
# Code just to increase the number of display units for the various components
import thermosteam as tmo
tmo.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
tmo.MultiStream.display_units.N = 40  
bst.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
bst.MultiStream.display_units.N = 40  

In [3]:
F.RCF_RXR2.results()

Hydrogenolysis reactor                                Units  RCF_RXR2
High pressure steam Duty                              kJ/hr  1.85e+07
                    Flow                            kmol/hr       575
                    Cost                             USD/hr       182
Design              Diameter                             ft      11.3
                    Length                               ft      33.9
                    Reactor volume                       m3      96.7
                    Total volume                         m3  2.61e+03
                    Residence time                       hr     0.333
                    Number of reactors                             27
                    Vessel type                              Vertical
                    Weight                               lb  3.49e+05
                    Wall thickness                       in      5.65
                    Duty                              kJ/hr  1.57e+07
                    Catalyst loading cost               USD  1.04e+05
Purchase cost       Vertical pressure vessel (x27)      USD  2.96e+07
                    Platform and ladders (x27)          USD  7.69e+05
Total purchase cost                                     USD  3.04e+07
Utility cost                                         USD/hr       182

In [4]:
F.RCF_RXR2

HydrogenolysisReactor: RCF_RXR2
ins...
[0] Solvolysis_Liquor  from  SolvolysisReactor-RCF_RXR1
    phases: ('g', 'l'), T: 498.15 K, P: 6.3e+06 Pa
    flow (kmol/hr): (g) Water          0.488
                        AceticAcid     3.56e-06
                        CO             83.8
                        Methanol       29.8
                        Hydrogen       9.16e-09
                        Methane        51.7
                    (l) Water          578
                        AceticAcid     48.6
                        Extract        7.4
                        SolubleLignin  87.9
                        Glucan         23.8
                        Xylan          5.92
                        Arabinan       0.757
                        Mannan         9.51
                        Galactan       3.6
                        Methanol       2e+04
                        Hydrogen       3.18e-19
                        Methane        0.00083
[1] s5  from  HXutility-RCF_HX2
    phases: ('g

In [5]:
chems['Methanol'].Psat(498.15)/101325

62.358525719925716

In [6]:
F.RCF_RXR2.ins[0]

MultiStream: Solvolysis_Liquor from <SolvolysisReactor: RCF_RXR1> to <HydrogenolysisReactor: RCF_RXR2>
phases: ('g', 'l'), T: 498.15 K, P: 6.3e+06 Pa
flow (kmol/hr): (g) Water          0.488
                    AceticAcid     3.56e-06
                    CO             83.8
                    Methanol       29.8
                    Hydrogen       9.16e-09
                    Methane        51.7
                (l) Water          578
                    AceticAcid     48.6
                    Extract        7.4
                    SolubleLignin  87.9
                    Glucan         23.8
                    Xylan          5.92
                    Arabinan       0.757
                    Mannan         9.51
                    Galactan       3.6
                    Methanol       2e+04
                    Hydrogen       3.18e-19
                    Methane        0.00083


In [7]:

from math import ceil
tau_residence = 1/3
void_frac = 0.7
free_frac  = 0.2
V_max_limit = 100
#### Volumetric flow and reactor volume sizing ########

# Total feed flow: liquid solvent/lignin (ins[0]) + hydrogen gas (ins[1])
Q_total = F.RCF_RXR2.ins[0].F_vol + F.RCF_RXR2.ins[1].F_vol        # [m³/hr]

# Fluid holdup in the catalyst bed voids
V_fluid = Q_total * tau_residence                 # [m³]

# Packed bed volume (catalyst particles + void space)
V_bed = V_fluid / void_frac                       # [m³]

# Total reactor volume: bed + free headspace above bed
V_reactor_total = V_bed / (1.0 - free_frac)      # [m³]

# Number of parallel reactors so each vessel stays within V_max_limit
N_reactors = max(1, ceil(V_reactor_total / V_max_limit))

In [8]:
Q_total

4385.550627282737

In [9]:
V_reactor_total

2610.4468019540104

In [10]:
N_reactors

27

In [11]:

tau_residence = 1/3
void_frac = 0.7
free_frac  = 0.2
V_max_limit = 100
#### Volumetric flow and reactor volume sizing ########

# Total feed flow: liquid solvent/lignin (ins[0]) + hydrogen gas (ins[1])
Q_total = F.RCF_RXR2.ins[0].F_vol + F.RCF_RXR2.ins[1].F_vol        # [m³/hr]

# Fluid holdup in the catalyst bed voids
V_fluid = Q_total * tau_residence                 # [m³]

# Packed bed volume (catalyst particles + void space)
V_bed = V_fluid / void_frac                       # [m³]

# Total reactor volume: bed + free headspace above bed
V_reactor_total = V_bed / (1.0 - free_frac)      # [m³]

# Number of parallel reactors so each vessel stays within V_max_limit
N_reactors = max(1, ceil(V_reactor_total / V_max_limit))
V_per_reactor = V_reactor_total / N_reactors           # [m³]
Q_per_reactor = Q_total / N_reactors                   # [m³/hr]

#### Reactor geometry ########

u = self.superficial_velocity
A = Q_per_reactor / (u * 3600)                        # [m²]
diameter = 2.0 * (A / np.pi) ** 0.5                   # [m]
length = V_per_reactor / A                             # [m]

# Enforce L/D within [LD_min, LD_max]; adjust u analytically if needed.
# From L/D = V/(A) / (2√(A/π)) → A = (V√π / (2·(L/D)))^(2/3)
LD = length / diameter
if LD > self.LD_max:
    A = (V_per_reactor * np.pi ** 0.5 / (2.0 * self.LD_max)) ** (2.0 / 3.0)
    u = Q_per_reactor / (A * 3600)
    self.superficial_velocity = u
    diameter = 2.0 * (A / np.pi) ** 0.5
    length = V_per_reactor / A
elif LD < self.LD_min:
    A = (V_per_reactor * np.pi ** 0.5 / (2.0 * self.LD_min)) ** (2.0 / 3.0)
    u = Q_per_reactor / (A * 3600)
    self.superficial_velocity = u
    diameter = 2.0 * (A / np.pi) ** 0.5
    length = V_per_reactor / A

self.area = A
self.diameter = diameter
self.length = length
self.N_reactors = N_reactors



NameError: name 'self' is not defined